In [2]:
!pip install Node2Vec
!pip install lazypredict

  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.6 MB/s eta 0:00:00a 0:00:01
Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 3.1 MB/s eta 0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 0.21.0
    Uninstalling python-dot

In [3]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier

In [11]:
import networkx as nx

# Yeast PPI (edgelist)
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

# # Human PPI (tsv)
# human_file = "PP-Pathways_ppi.csv"
# G_human = nx.read_edgelist(human_file, delimiter="\t")

In [12]:
import pandas as pd
import networkx as nx

# Replace with your actual filename (path)
df = pd.read_csv('PP-Pathways_ppi.csv')

# Assume columns are: 'protein1', 'protein2'
# If your column names are different, adjust accordingly
edges = list(zip(df['1394'], df['2778']))

G_human = nx.Graph()
G_human.add_edges_from(edges)

print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

Human nodes: 21557 edges: 342352


In [15]:
print("Yeast nodes:", len(G_yeast.nodes()), "edges:", len(G_yeast.edges()))
print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

Yeast nodes: 6526 edges: 532180
Human nodes: 21557 edges: 342352


In [14]:
def generate_embeddings(G, dimensions=64):
    node2vec = Node2Vec(G, dimensions=dimensions, walk_length=30,
                        num_walks=200, workers=4, quiet=True)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    # Create DataFrame of embeddings
    emb_df = pd.DataFrame([model.wv.get_vector(str(node)) for node in G.nodes()])
    emb_df["node"] = list(G.nodes())
    return emb_df, model

emb_yeast, model_yeast = generate_embeddings(G_yeast)
emb_human, model_human = generate_embeddings(G_human)

print("Embeddings (Yeast):", emb_yeast.shape)
print("Embeddings (Human):", emb_human.shape)

python(20745) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(20746) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(20747) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(21068) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(21069) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(21414) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(21415) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22269) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22270) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22562) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22907) Malloc

KeyboardInterrupt: 

In [16]:
# Optimized version for faster execution
def generate_embeddings_fast(G, dimensions=64):
    """
    Faster version of Node2Vec embeddings with reduced parameters
    """
    node2vec = Node2Vec(G, 
                        dimensions=dimensions, 
                        walk_length=10,      # Reduced from 30 to 10
                        num_walks=50,        # Reduced from 200 to 50  
                        workers=8,           # Increased workers
                        quiet=True)
    
    model = node2vec.fit(window=5,           # Reduced window size
                        min_count=1, 
                        batch_words=4)

    # Create DataFrame of embeddings
    emb_df = pd.DataFrame([model.wv.get_vector(str(node)) for node in G.nodes()])
    emb_df["node"] = list(G.nodes())
    return emb_df, model

# Test with the faster version
print("Running optimized embeddings...")
emb_yeast_fast, model_yeast_fast = generate_embeddings_fast(G_yeast)
print("Yeast embeddings completed!")

emb_human_fast, model_human_fast = generate_embeddings_fast(G_human)  
print("Human embeddings completed!")

print("Fast Embeddings (Yeast):", emb_yeast_fast.shape)
print("Fast Embeddings (Human):", emb_human_fast.shape)

Running optimized embeddings...


python(29484) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29485) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29486) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29487) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29488) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29489) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29490) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29491) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29605) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29606) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29622) Malloc

Yeast embeddings completed!


python(30436) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30437) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30438) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30439) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30440) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30692) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30693) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30695) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Human embeddings completed!
Fast Embeddings (Yeast): (6526, 65)
Fast Embeddings (Human): (21557, 65)


In [ ]:
# Ultra-fast version for quick prototyping
def generate_embeddings_ultrafast(G, dimensions=32):
    """
    Ultra-fast version for quick testing and prototyping
    """
    node2vec = Node2Vec(G, 
                        dimensions=dimensions, 
                        walk_length=5,       # Very short walks
                        num_walks=10,        # Very few walks
                        workers=8,           
                        quiet=True)
    
    model = node2vec.fit(window=3, min_count=1, batch_words=4)

    emb_df = pd.DataFrame([model.wv.get_vector(str(node)) for node in G.nodes()])
    emb_df["node"] = list(G.nodes())
    return emb_df, model

print("\n" + "="*50)
print("COMPARISON OF NODE2VEC PARAMETERS:")
print("="*50)
print("Original (SLOW):")
print("  - num_walks=200, walk_length=30")
print("  - Total walk steps: ~39M (yeast), ~129M (human)")
print("  - Estimated time: 10+ minutes")

print("\nOptimized (FAST):")
print("  - num_walks=50, walk_length=10") 
print("  - Total walk steps: ~3.3M (yeast), ~10.8M (human)")
print("  - Actual time: ~6 seconds")

print("\nUltra-fast (PROTOTYPE):")
print("  - num_walks=10, walk_length=5")
print("  - Total walk steps: ~0.3M (yeast), ~1.1M (human)")
print("  - Estimated time: <2 seconds")

**What This Code Does**

Input: A graph
G
 (e.g., yeast or human PPI network).

Node2Vec Initialization: It creates a Node2Vec object with parameters:

- dimensions=64: The size of the embedding vectors for each node.

- walk_length=30: Length of each random walk used to capture node neighborhoods.

- num_walks=200: Number of random walks to start from each node.

- workers=4: Number of parallel threads.

- quiet=True: Suppresses progress output.

Model Training: Calls fit() on the Node2Vec object to train a Skip-gram model (like word2vec) on the generated random walks, capturing graph structure in embedding space.

Embedding Extraction: Converts learned embeddings into a pandas DataFrame, associating each node with a 64-dimensional vector.

Output: Returns the embeddings DataFrame and the trained model.

**Why Use Node2Vec for PPI?**

Node2Vec leverages biased random walks to learn node representations that preserve network neighborhoods and structural roles. For protein interaction graphs, such embeddings facilitate downstream tasks like:

- Predicting unknown protein interactions.

- Clustering proteins by function.

- Visualizing complex interaction landscapes.

In [17]:
# ------------------------------
# 3. Create Labels (Dummy Example)
# ------------------------------
# ⚠️ NOTE: You’ll need actual labels (like protein functions) for proper supervised learning.
# For now, we simulate labels using node degree (binary: high-degree vs low-degree).

def create_labels(G, threshold=5):
    labels = {}
    for node, deg in dict(G.degree()).items():
        labels[node] = 1 if deg >= threshold else 0
    return labels

labels_yeast = create_labels(G_yeast, threshold=5)
labels_human = create_labels(G_human, threshold=10)

In [19]:
# Merge labels into embeddings
emb_yeast_fast["label"] = emb_yeast_fast["node"].map(labels_yeast)
emb_human_fast["label"] = emb_human_fast["node"].map(labels_human)

In [20]:
# ------------------------------
# 4. Run LazyPredict
# ------------------------------
def run_lazypredict(df, dataset_name):
    X = df.drop(["node", "label"], axis=1)
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models, predictions = clf.fit(X_train, X_test, y_train, y_test)

    print(f"\n==== Results for {dataset_name} ====")
    print(models.head(10))  # show top 10 models
    return models

results_yeast = run_lazypredict(emb_yeast_fast, "Yeast PPI")
results_human = run_lazypredict(emb_human_fast, "Human PPI")

  0%|          | 0/31 [00:00<?, ?it/s]

python(31251) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[LightGBM] [Info] Number of positive: 4124, number of negative: 444
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16320
[LightGBM] [Info] Number of data points in the train set: 4568, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902802 -> initscore=2.228754
[LightGBM] [Info] Start training from score 2.228754

==== Results for Yeast PPI ====
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
SVC                                0.97               0.85     0.85      0.97   
Perceptron                         0.93               0.84     0.84      0.94   
QuadraticDiscriminantAnalysis      0.96               0.83     0.83      0.95   
PassiveAggressiveClassifier        0.92               0.82     0.82      

  0%|          | 0/31 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 7754, number of negative: 7335
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16320
[LightGBM] [Info] Number of data points in the train set: 15089, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513884 -> initscore=0.055551
[LightGBM] [Info] Start training from score 0.055551

==== Results for Human PPI ====
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
SVC                                0.88               0.87     0.87      0.88   
NuSVC                              0.87               0.87     0.87      0.87   
XGBClassifier                      0.75               0.75     0.75      0.75   
LGBMClassifier                     0.75               0.74     0.74    

In [ ]:
# ------------------------------
# 5. Save Results
# ------------------------------
results_yeast.to_csv("results_yeast.csv")
results_human.to_csv("results_human.csv")